# 01 — Training: CV <-> Job Posting Skill-Gap — NER pipeline (DistilBERT)

Fine-tunes a pretrained DistilBERT token classifier to extract skill spans (TECH / SOFT / TOOL / DOMAIN / CERT) from job postings, using distant supervision from the postings' existing skill annotations.

**This notebook only covers training.** Building the role database and running the CV-vs-role comparison happens in `02_experiments.ipynb`; smoke-testing the saved artifacts happens in `03_testing.ipynb`.

> **Network note:** fine-tuning `distilbert-base-uncased` requires downloading its pretrained weights and tokenizer from `huggingface.co` the first time. Run this notebook somewhere with internet access (local machine, Colab, Kaggle, a training server); pre-download once with `huggingface-cli download distilbert-base-uncased`. A GPU is strongly recommended.

**Output of this notebook:** a fine-tuned checkpoint saved to `skill_ner_distilbert_best/` (best validation F1), ready to be picked up by `02_experiments.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import re
import json
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)


Device: cuda


## Stage 1 — Load job postings

Each posting has skills split into five categories (`skills_technical`, `skills_soft`, `skills_tool`, `skills_domain`, `skills_certification`). We keep the category split — it becomes the entity *type* in the NER tag set.

In [2]:
df = pd.read_csv('../data/processed/data_tech_only.csv')
df = df[df["skill_count"] > 0].dropna(subset=["cleaned_text"]).reset_index(drop=True)
print(df.shape)
df.head()


(6850, 11)


,row_index,title,cleaned_text,skills_technical,skills_soft,skills_tool,skills_domain,skills_certification,all_skills,skill_count,cleaned_title
0,0,10 + Blockchain Nodes / Masternodes to set up,blockchain nodes masternodes set requirements ...,blockchain; Kyber Network; Nebulas; SecretNetw...,NaN,NaN,crypto,NaN,blockchain; crypto; Kyber Network; Nebulas; Se...,16,blockchain nodes masternodes set
1,1,10 .NET Developers (Middle and Senior level),net developers middle senior level greetings n...,.NET; microservices; multi-cloud; SaaS,NaN,NaN,NaN,NaN,.NET; microservices; multi-cloud; SaaS,4,net developers middle senior level
2,2,"10X Engineer (co-founder, #4 employee, USD 11-...",x engineer co founder employee usd k equity pr...,live video chat; co-browsing,NaN,NaN,NaN,NaN,live video chat; co-browsing,2,x engineer co founder employee usd k equity
3,5,1806/01 PHP Developer,php developer requirements : experience php ye...,PHP; Laravel; JavaScript; Vue.js; MySQL; serve...,NaN,Git,NaN,NaN,PHP; Laravel; JavaScript; Vue.js; MySQL; Git; ...,7,php developer
4,9,1812/35 Middle DataBase developer,middle database developer job responsibilities...,SQL; database development; stored procedures; ...,NaN,SQL Profiler,NaN,NaN,SQL; database development; stored procedures; ...,7,middle database developer


## Stage 2 — Build token-level BIO tags (distant supervision)

Match each known skill phrase from the annotation columns onto a word span in `cleaned_text`, longest phrase first, so the tagger has ground-truth labels to train against.

In [3]:
CATEGORY_COLS = {
    "TECH": "skills_technical",
    "SOFT": "skills_soft",
    "TOOL": "skills_tool",
    "DOMAIN": "skills_domain",
    "CERT": "skills_certification",
}

def parse_skills(cell):
    if pd.isna(cell):
        return []
    return [s.strip() for s in str(cell).split(";") if s.strip()]

for col in CATEGORY_COLS.values():
    df[col + "_list"] = df[col].apply(parse_skills)

def tokenize(text):
    # Word-level tokenizer used only to build ground-truth BIO tags (not fed to the model).
    return re.findall(r"[a-z0-9+#.]+", str(text).lower())

TAG_LIST = ["O"]
for cat in CATEGORY_COLS:
    TAG_LIST += [f"B-{cat}", f"I-{cat}"]
tag_to_idx = {t: i for i, t in enumerate(TAG_LIST)}
id2label = {i: t for i, t in enumerate(TAG_LIST)}
label2id = {t: i for i, t in enumerate(TAG_LIST)}
O_IDX = tag_to_idx["O"]
NUM_TAGS = len(TAG_LIST)
print("Tags:", TAG_LIST)

def find_span(tokens, phrase_tokens, taken):
    n, m = len(tokens), len(phrase_tokens)
    if m == 0:
        return None
    for i in range(n - m + 1):
        if any(taken[i:i + m]):
            continue
        if tokens[i:i + m] == phrase_tokens:
            return i
    return None

def build_bio_tags(text, skills_by_cat):
    # Distant-supervision BIO tagging: match each known skill phrase to a word span.
    tokens = tokenize(text)
    tags = [O_IDX] * len(tokens)
    taken = [False] * len(tokens)

    ordered = [(cat, s) for cat, skills in skills_by_cat.items() for s in skills]
    ordered.sort(key=lambda x: -len(tokenize(x[1])))  # longest phrase first

    for cat, skill in ordered:
        phrase_tokens = tokenize(skill)
        start = find_span(tokens, phrase_tokens, taken)
        if start is None:
            continue
        end = start + len(phrase_tokens)
        tags[start] = tag_to_idx[f"B-{cat}"]
        for i in range(start + 1, end):
            tags[i] = tag_to_idx[f"I-{cat}"]
        for i in range(start, end):
            taken[i] = True
    return tokens, tags

all_tokens, all_tags = [], []
for _, row in df.iterrows():
    skills_by_cat = {cat: row[col + "_list"] for cat, col in CATEGORY_COLS.items()}
    toks, tags = build_bio_tags(row["cleaned_text"], skills_by_cat)
    all_tokens.append(toks)
    all_tags.append(tags)

df["tokens"] = all_tokens
df["bio_tags"] = all_tags

covered = sum(any(t != O_IDX for t in tags) for tags in all_tags)
print(f"Postings with >=1 tagged skill span: {covered}/{len(df)}")


Tags: ['O', 'B-TECH', 'I-TECH', 'B-SOFT', 'I-SOFT', 'B-TOOL', 'I-TOOL', 'B-DOMAIN', 'I-DOMAIN', 'B-CERT', 'I-CERT']
Postings with >=1 tagged skill span: 6205/6850


## Stage 3 — Train / validation / test split

In [4]:
idx_train, idx_temp = train_test_split(df.index, test_size=0.3, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)
print(f"train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}")


train: 4795  val: 1027  test: 1028


## Stage 4 — Tokenizer + dataset

Downloads `distilbert-base-uncased` from huggingface.co on first run.

In [5]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 100
PAD_TAG = -100  # ignored by the loss

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception as e:
    print(
        "Could not download the pretrained tokenizer/model from huggingface.co.\n"
        "This environment has no network access to huggingface.co, so this is expected here.\n"
        "Run this notebook somewhere with internet access (local machine, Colab, Kaggle, a "
        "training server) -- it will download distilbert-base-uncased automatically, or "
        "pre-download it once with:  huggingface-cli download distilbert-base-uncased\n"
    )
    raise

def align_labels_with_tokens(word_ids, word_tags):
    labels = []
    prev_word_id = None
    for wid in word_ids:
        if wid is None:
            labels.append(PAD_TAG)
        elif wid != prev_word_id:
            labels.append(word_tags[wid])   # first subword of this word gets the real tag
        else:
            labels.append(PAD_TAG)          # continuation subwords are ignored by the loss
        prev_word_id = wid
    return labels


In [6]:
class SkillNERDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = df.loc[self.indices[i]]
        enc = tokenizer(
            row["tokens"], is_split_into_words=True, truncation=True,
            max_length=MAX_LEN, padding="max_length", return_tensors="pt",
        )
        word_ids = enc.word_ids(batch_index=0)
        labels = align_labels_with_tokens(word_ids, row["bio_tags"])
        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "labels": torch.tensor(labels, dtype=torch.long),
        }

BATCH_SIZE = 16
train_loader = DataLoader(SkillNERDataset(idx_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SkillNERDataset(idx_val), batch_size=BATCH_SIZE)
test_loader  = DataLoader(SkillNERDataset(idx_test), batch_size=BATCH_SIZE)


## Stage 5 — Model

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_TAGS, id2label=id2label, label2id=label2id,
).to(device)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 66,371,339


## Stage 6 — Evaluation helpers (entity-level precision/recall/F1)

In [8]:
def tags_to_spans(tags):
    spans = []
    start, cat = None, None
    for i, t in enumerate(list(tags) + [O_IDX]):
        label = TAG_LIST[t] if t != PAD_TAG else "O"
        if label.startswith("B-"):
            if start is not None:
                spans.append((start, i, cat))
            start, cat = i, label[2:]
        elif label.startswith("I-") and cat == label[2:]:
            continue
        else:
            if start is not None:
                spans.append((start, i, cat))
            start, cat = None, None
    return spans

def entity_prf(all_pred_tags, all_true_tags):
    tp = fp = fn = 0
    for pred, true in zip(all_pred_tags, all_true_tags):
        pred_spans = set(tags_to_spans(pred))
        true_spans = set(tags_to_spans(true))
        tp += len(pred_spans & true_spans)
        fp += len(pred_spans - true_spans)
        fn += len(true_spans - pred_spans)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


## Stage 7 — Train

Saves the best-validation-F1 checkpoint to `skill_ner_distilbert_best/` (overwritten each time validation F1 improves), with early stopping.

In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n_examples = 0.0, 0
    pred_seqs, true_seqs = [], []

    with torch.set_grad_enabled(train):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * input_ids.size(0)
            n_examples += input_ids.size(0)

            preds = outputs.logits.argmax(-1)
            for p_row, y_row in zip(preds.cpu().tolist(), labels.cpu().tolist()):
                idxs = [j for j, t in enumerate(y_row) if t != PAD_TAG]
                pred_seqs.append([p_row[j] for j in idxs])
                true_seqs.append([y_row[j] for j in idxs])

    precision, recall, f1 = entity_prf(pred_seqs, true_seqs)
    return total_loss / max(n_examples, 1), precision, recall, f1


EPOCHS = 4
PATIENCE = 2
best_val_f1 = -1
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, *_ = run_epoch(train_loader, train=True)
    val_loss, val_p, val_r, val_f1 = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | "
          f"val_loss {val_loss:.4f} val_precision {val_p:.3f} val_recall {val_r:.3f} val_f1 {val_f1:.3f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        model.save_pretrained("../Models/skill_ner_distilbert_best")
        tokenizer.save_pretrained("../Models/skill_ner_distilbert_best")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping — validation F1 stopped improving.")
            break


Epoch 01 | train_loss 0.1877 | val_loss 0.2126 val_precision 0.689 val_recall 0.610 val_f1 0.647


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 02 | train_loss 0.1562 | val_loss 0.2045 val_precision 0.678 val_recall 0.637 val_f1 0.657


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 03 | train_loss 0.1278 | val_loss 0.2154 val_precision 0.722 val_recall 0.582 val_f1 0.644
Epoch 04 | train_loss 0.1043 | val_loss 0.2265 val_precision 0.717 val_recall 0.611 val_f1 0.660


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Stage 8 — Reload best checkpoint and evaluate on the held-out test set

In [11]:
model = AutoModelForTokenClassification.from_pretrained("../Models/skill_ner_distilbert_best").to(device)
test_loss, test_p, test_r, test_f1 = run_epoch(test_loader, train=False)
print(f"TEST — precision {test_p:.3f} | recall {test_r:.3f} | f1 {test_f1:.3f}")


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

TEST — precision 0.720 | recall 0.620 | f1 0.666


Training is done here. `skill_ner_distilbert_best/` on disk now holds the fine-tuned model + tokenizer — pick it up from `02_experiments.ipynb` to build the role database and try it on CVs.